# Dataset du trafic routier

## Key takeaways

**En bref.** Trafic routier horaire de Paris (comptages permanents), de novembre 2025 à septembre 2026. Après nettoyage, on garde **1 531 arcs** qui ont un débit pour au moins 70 % des heures, sur une grille complète de **8 016 heures** : 12 272 496 lignes, avec des `NaN` là où la mesure manque (8,6 % des débits, 23,2 % des taux d'occupation). Le dataset est produit par `uv run python -m src.clean.trafic` dans `data/interim/trafic/trafic.parquet`, avec à côté `data/interim/trafic/referentiel_nettoye.parquet` (tracé et nœuds des 1 531 arcs, pour les cartes). Ce notebook refait les mêmes étapes en les commentant.

### Données brutes
- 11 fichiers mensuels, chacun avec exactement les heures qui commencent dans le mois : ≈2M lignes par mois et ≈23M au total (23 489 735 lignes, 2 980 arcs). Les mois sont comparables (presque le même nombre d'arcs, même part de débits manquants) : on peut les concaténer.
- Colonnes : `iu_ac` (identifiant de l'arc), `t_1h` (heure UTC, **fin** de période), `q` (débit, véh/h), `k` (taux d'occupation, %), `etat_trafic` (déduit de `k`), `etat_barre` (Ouvert / Barré / Invalide).
- Un référentiel (3 739 lignes) donne le tracé de chaque arc, ses nœuds amont et aval, et des dates de validité.

### Nettoyage (dans l'ordre du script)
| étape | ce qu'on fait | ce qui est retiré |
|---|---|---|
| préparation | on garde `iu_ac`, `q`, `k` ; `t_1h` devient `heure` (début de période, UTC) ; on ne garde que les heures de la période | aucune ligne |
| référentiel | certains arcs y sont en doublon (mises à jour de tracé ou de capteur) : on garde la version dont `date_fin`, puis `date_debut`, est la plus récente | 391 lignes du référentiel sur 3 739 (10,5 %) |
| arcs sans géométrie | certains arcs du trafic n'ont pas de position dans le référentiel (exemple : avenue de la Porte de Clichy) : on les supprime | 31 arcs (1,0 %), dont 21 avec des mesures ; 234 126 lignes (1,0 %) |
| zéros consécutifs | plus de 24 h de suite à zéro = capteur en panne, valeurs mises à `NaN` (les lignes restent) | 6 049 débits (0,05 %), 6 799 taux (0,05 %) |
| seuil de 70 % | on garde les arcs qui ont un débit pour au moins 70 % des heures | 1 418 arcs (48,1 %), 11 182 457 lignes ; mais seulement 11,0 % des débits renseignés |
| grille complète | une ligne par arc et par heure ; les heures absentes sont ajoutées avec `NaN` | + 199 344 lignes ajoutées (1,7 %) |
| infos des arcs | libellé, longueur et centroïde de l'arc sur chaque ligne | aucune ligne |

Le détail, avec les pourcentages à chaque étape, est dans le **bilan du nettoyage** en fin de notebook.

### Pourquoi se concentrer sur le débit
Tous les arcs n'ont pas les données de débit et de taux d'occupation, et parmi ceux qui les ont, tous ne les ont pas pour toutes les heures.

Le débit seul ne suffit pas à décrire le trafic, car un débit faible (300 véhicules par heure) peut correspondre à une rue très peu fréquentée ou bouchée. Cependant, pour la corrélation aux émissions de particules, le taux peut être omis en première approximation. Les émissions s'écrivent :
$$ E ≈ Q \times L \times FE(v)$$

où $Q$ est le débit, $L$ la longueur de l'axe et $FE(v)$ le facteur d'émissions par kilomètre. Ainsi, un axe bouché à 300 v/h émet peu malgré un grand taux d'occupation, car les voitures font peu de distance. Certes, les véhicules émettent 2 à 3 fois plus à faible vitesse et à cause des accélérations et freinages dus aux embouteillages ; cependant, cet effet peut être dans un premier temps négligé à côté des variations de débit (d'un facteur 20). De plus, il tend à se réduire avec le 
_start-and-stop_.

Ainsi, on peut se concentrer d'abord sur les axes avec des données de débit, et au vu de la distribution des données (bimodale : des arcs presque toujours mesurés, d'autres jamais), nous choisissons de garder uniquement les axes avec des relevés de débit pour plus de 70 % des heures de l'année.

### Dataset final
- Colonnes : `iu_ac`, `heure`, `q`, `k`, `libelle`, `longueur_m`, `x_centroide`, `y_centroide` (centroïde en Lambert-93, EPSG:2154, en mètres).
- Absents : la forme des arcs (réduite à son centroïde), les nœuds amont et aval, `etat_trafic` et `etat_barre`. On les retrouve dans `data/interim/trafic/referentiel_nettoye.parquet`, déjà dédoublonné, par `iu_ac`.
- Période couverte : du 01/11/2025 00:00 au 30/09/2026 23:00 UTC (heures de début de période).

### Ce que montrent les données
- Débit : médiane 297 véh/h, moyenne 618, écart-type 1 040, quartiles 145 et 564, maximum 11 793. La plupart des débits sont sous 2 000 véh/h, avec un second groupe vers 5 000 véh/h (périphérique).
- Les arcs gardés mesurent 174 m en médiane (de 7 à 871 m).
- Le trafic suit le rythme des journées : creux la nuit, plus faible le week-end.
- Le périphérique et les grands axes concentrent le trafic ; le taux d'occupation est le plus fort sur le périphérique et quelques carrefours.

### Points d'attention
- Il reste des valeurs à `NaN` : on obtient une valeur de débit par arc et par heure, ! éventuellement égale à NaN !, car toutes les valeurs ne sont pas présentes (certaines heures et journées sont absentes du dataset). Rien n'est imputé à ce stade.
- Les heures ont été modifiées pour correspondre au **début** de l'heure considérée, et sont en **UTC**.
- Heures incomplètes dans le dataset final (1 531 arcs, 8 016 heures) :
  - **débit** : 100 % des heures ont au moins un arc sans débit, et **18,6 %** des heures ont plus de 10 % des arcs sans débit ; en médiane, 5,9 % des arcs n'ont pas de débit à une heure donnée ;
  - **taux d'occupation** : 100 % des heures ont plus de 10 % des arcs sans `k`, car les arcs sont choisis sur le débit et 150 d'entre eux n'ont jamais de `k`.
- Cinq coupures de la collecte (29/12/2025, 29/01/2026, 16/02/2026, 24/03/2026, 13/04/2026) vident presque toutes les mesures d'une journée ; hors coupures, il manque 6,2 % des débits un jour médian.
- Les dates de validité du référentiel sont toutes expirées avant la période : elles ne servent qu'à choisir entre les versions d'un arc. Deux arcs (VERG, Souterrain_Etoile) ont deux versions aux mêmes dates, sans effet car ils n'ont aucune donnée de trafic.
- Les zéros des rues barrées sont gardés : les longues séries de zéros sont à 98,8 % sur des arcs « Ouvert ». Limite : une bretelle fermée plusieurs jours pour travaux sans être signalée « Barré » serait prise pour une panne.

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.lines import Line2D

sys.path.append("..")  # pour importer src depuis le dossier notebooks/
from src import stockage
from src.clean import trafic as nettoyage
from src.extract import trafic as extraction
from src.extract.trafic import RACINE, lire_config

# les fonctions de nettoyage journalisent le nombre de lignes concernées : on les affiche
logging.basicConfig(level=logging.INFO, format="%(message)s")

CFG = lire_config()
SEUIL_DEBIT = CFG["trafic"]["part_sans_debit_max"]
SEUIL_ZEROS = CFG["trafic"]["zeros_consecutifs_max"]
HEURES = nettoyage.toutes_les_heures(CFG["periode"])  # heures de la période, début de période en UTC

BLEU, GRIS, ROUGE = "#2a78d6", "#a8a7a2", "#d62728"
MOIS = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]
JOURS = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e5e1",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 9,
    "font.size": 9,
})

## Vue d'ensemble

Par défaut, les fichiers bruts sont récupérés depuis SSP Cloud, où ils sont figés : l'API ne garde que 13 mois glissants, une nouvelle extraction ne donnerait donc plus les mêmes mois. Pour repartir de la source, passer `RELANCER_EXTRACTION` à `True` (environ 1 h la première fois, avec un cache par mois).

In [ ]:
RELANCER_EXTRACTION = False

if RELANCER_EXTRACTION:
    extraction.main()
elif not (RACINE / CFG["chemins"]["raw"] / "trafic").exists():
    stockage.recuperer(CFG, dataset="trafic")

In [ ]:
# Lignes et distribution par mois pour justifier la jointure
# nom des colonnes
trafic_brut, ref = nettoyage.charger_brut(RACINE / CFG["chemins"]["raw"] / "trafic")

descriptions = {
    "iu_ac": "identifiant de l'arc, clé de jointure avec le référentiel",
    "t_1h": "heure UTC, fin de la période d'une heure",
    "q": "débit (véhicules par heure)",
    "k": "taux d'occupation (%)",
    "etat_trafic": "Fluide / Pré-saturé / Saturé / Bloqué, déduit de k",
    "etat_barre": "Ouvert / Barré / Invalide",
}
display(pd.DataFrame({"type": trafic_brut.dtypes.astype(str), "description": pd.Series(descriptions)}))
print("Colonnes du référentiel :", ref.columns.tolist())

# t_1h marque la fin de l'heure : le mois est celui du début de l'heure (le mois du fichier)
mois = (trafic_brut["t_1h"] - pd.Timedelta(hours=1)).dt.strftime("%Y-%m")
par_mois = trafic_brut.groupby(mois).agg(lignes=("iu_ac", "size"), arcs=("iu_ac", "nunique"),
                                         heures=("t_1h", "nunique"))
par_mois["part_sans_debit"] = trafic_brut["q"].isna().groupby(mois).mean().round(3)
par_mois.index.name = "mois"
par_mois

Les 11 mois ont presque le même nombre d'arcs (de 2 977 à 2 979) et une part de débits manquants proche : on peut les concaténer. Les quelques heures en moins certains mois sont des coupures de la collecte (journées entières, en heure de Paris, absentes pour tous les arcs).

## Nettoyage

In [ ]:
# Garder les axes avec date_fin la plus récente dans le référentiel
# Supprimer les axes sans données géo
# preparer : t_1h marque la fin de la période, la colonne heure marque son début (convention du projet) ;
# seules les heures de la période sont gardées
def resume(df, etape):
    """Lignes, arcs et valeurs renseignées après une étape (pour le bilan en fin de notebook)."""
    return {"étape": etape, "lignes": len(df), "arcs": df["iu_ac"].nunique(),
            "débits q renseignés": df["q"].notna().sum(), "taux k renseignés": df["k"].notna().sum()}


trafic = nettoyage.preparer(trafic_brut, HEURES)
etapes = [resume(trafic, "données brutes")]
arcs = nettoyage.dedoublonner_referentiel(ref)

sans_geo = trafic[~trafic["iu_ac"].isin(arcs["iu_ac"])]
trafic = nettoyage.retirer_arcs_sans_geo(trafic, arcs)
etapes.append(resume(trafic, "arcs sans géométrie"))
print("dont arcs avec au moins une mesure (q ou k) :", sans_geo.dropna(subset=["q", "k"], how="all")["iu_ac"].nunique())

In [ ]:
# Analyse des zéros ou 100%: retirer plus de 24h de zéros consécutifs
print("Lignes avec q = 0 :", (trafic["q"] == 0).sum(), "| k = 0 :", (trafic["k"] == 0).sum(),
      "| k >= 100 % :", (trafic["k"] >= 100).sum())
libelles = arcs.set_index("iu_ac")["libelle"]
for colonne in ["q", "k"]:
    plus_longue = nettoyage.longueur_series_de_zeros(trafic, colonne).groupby(trafic["iu_ac"]).max()
    print(f"{colonne} : plus longue série de zéros {plus_longue.max()} h ({libelles[plus_longue.idxmax()]}) ; "
          f"{(plus_longue > 0).sum()} arcs ont au moins un zéro")

for colonne in ["q", "k"]:
    trafic = nettoyage.retirer_longues_series_de_zeros(trafic, colonne, SEUIL_ZEROS)
etapes.append(resume(trafic, f"séries de zéros de plus de {SEUIL_ZEROS} h"))

Aucun taux d'occupation n'atteint 100 %. Les zéros isolés ou courts sont de vrais zéros (rues vides la nuit) ; au-delà de 24 h de suite, une rue vide est invraisemblable : on considère le capteur en panne et on met ces valeurs à `NaN`.

Faut-il épargner les zéros des rues barrées (`etat_barre = Barré`) ? Non : 98,8 % des débits nuls dans des séries de plus de 24 h sont sur des arcs « Ouvert », et aucune de ces séries n'est entièrement « Barré ». Les fermetures produisent surtout des zéros courts (moins de 24 h), que la règle garde déjà comme de vrais zéros. Seule limite : une bretelle fermée plusieurs jours pour travaux sans être signalée « Barré » serait prise pour une panne.

### Axes conservés

In [ ]:
# Montrer la distribution du débit manquant en couleur par arc et jour
# ditribution du nombre d'arc par part d'heures manquantes dans l'arc avec une ligne rouge verticale au niveau du seuil retenu (30%)
part = nettoyage.part_heures_sans_debit(trafic, HEURES).sort_values()
n_gardes = (part <= SEUIL_DEBIT).sum()

# part des 24 heures de chaque jour où le débit est renseigné, par arc (arcs triés comme part)
jour = trafic["heure"].dt.floor("D")
jours = pd.date_range(jour.min(), jour.max(), freq="D")
dispo = trafic["q"].notna().groupby([trafic["iu_ac"], jour]).sum().unstack(fill_value=0) # groupby par un multiindex en produit cartésien
# unstack repasse l'index jour en colonne, mettant 0 au lieu de NaN pour les couples (arc, jour) n'existant pas
dispo = dispo.reindex(index=part.index, columns=jours, fill_value=0) / 24
# part.index correspond aux iu_ac en ordre de part de debit manquant; 
# ajoute une ligne s'il y a un arc absent de dispo mais présent dans part (supprime si l'inverse)
# ajoute une colonne s'il  y a un jour manquant

debuts_mois = np.where(jours.day == 1)[0]
noms_mois = [f"{MOIS[d.month - 1]} {d.year % 100}" for d in jours[debuts_mois]]

fig, ax = plt.subplots(figsize=(11, 5))
cmap_dispo = LinearSegmentedColormap.from_list("dispo", ["#f3f2ee", BLEU, "#08306b"])
img = ax.imshow(dispo.values, aspect="auto", cmap=cmap_dispo, vmin=0, vmax=1, interpolation="nearest")
ax.axhline(n_gardes - 0.5, color=ROUGE, linewidth=1.5, label=f"seuil : {n_gardes} arcs gardés au-dessus")
ax.legend(loc="lower left", frameon=False)
ax.set_xticks(debuts_mois, noms_mois)
ax.set_xlabel("jour (UTC)")
ax.set_ylabel("arcs, triés par part d'heures sans débit")
ax.set_title("Le débit manque surtout arc par arc : beaucoup d'arcs n'en ont jamais, d'autres presque toujours")
ax.grid(False)
fig.colorbar(img, ax=ax, shrink=0.7, pad=0.02, label="part des 24 h du jour avec un débit")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(part * 100, bins=50, color=BLEU)
ax.axvline(SEUIL_DEBIT * 100, color=ROUGE, linewidth=1.5, label=f"seuil retenu : {SEUIL_DEBIT:.0%}")
ax.set_title(f"{n_gardes} arcs sur {len(part)} ont un débit pour au moins {1 - SEUIL_DEBIT:.0%} des heures")
ax.set_xlabel("part d'heures sans débit sur la période (%)")
ax.set_ylabel("nombre d'arcs")
ax.legend(frameon=False)
plt.show()

trafic = nettoyage.garder_arcs_mesures(trafic, SEUIL_DEBIT, HEURES)
etapes.append(resume(trafic, f"arcs avec plus de {SEUIL_DEBIT:.0%} d'heures sans débit"))

In [ ]:
# Visualisation des actes conservés sur tous les axes en traçant la figure geopandas
arcs_gardes = arcs[arcs["iu_ac"].isin(trafic["iu_ac"])]

fig, ax = plt.subplots(figsize=(8, 7))
arcs.plot(ax=ax, color=GRIS, linewidth=0.5)
arcs_gardes.plot(ax=ax, color=BLEU, linewidth=1.2)
ax.legend(handles=[Line2D([0], [0], color=BLEU, lw=2, label=f"arcs gardés ({len(arcs_gardes)})"),
                   Line2D([0], [0], color=GRIS, lw=2, label=f"autres arcs du référentiel ({len(arcs) - len(arcs_gardes)})")],
          loc="lower left", frameon=False)
ax.set_title("Les arcs gardés couvrent tout Paris, périphérique compris")
ax.set_axis_off()
plt.show()

In [ ]:
# ! Attention, à ce niveau, le dataset doit être dans son état final: une ligne par heure pour chaque axe, éventuellement égale à NaN. Si une heure n'était pas présente, il faut la rajouter (avec NaN comme valeur de taux d'occupation et de débit pour tous les axes)
# Même enchaînement que `uv run python -m src.clean.trafic`, qui écrit le résultat dans data/interim/trafic/trafic.parquet
trafic = nettoyage.completer_grille(trafic, HEURES)
etapes.append(resume(trafic, "grille complète"))
# libellé, longueur et centroïde de l'arc (Lambert-93, en mètres) sur chaque ligne
trafic = nettoyage.ajouter_infos_arcs(trafic, arcs)

n_arcs, n_heures = trafic["iu_ac"].nunique(), trafic["heure"].nunique()
assert len(trafic) == n_arcs * n_heures
assert not trafic.duplicated(["iu_ac", "heure"]).any()
print(f"{n_arcs} arcs x {n_heures} heures = {len(trafic)} lignes, de {trafic['heure'].min()} à {trafic['heure'].max()}")
print(f"Débit manquant : {trafic['q'].isna().mean():.1%} | taux d'occupation manquant : {trafic['k'].isna().mean():.1%}")
trafic.head()

Le dataset final (`data/interim/trafic/trafic.parquet`) contient, pour chaque arc et chaque heure : `iu_ac`, `heure`, `q`, `k`, `libelle`, `longueur_m`, `x_centroide` et `y_centroide`. Il ne contient **pas** la forme des arcs (le tracé complet est réduit à son centroïde), ni les nœuds amont et aval, ni `etat_trafic` et `etat_barre`. Ces informations sont enregistrées à part dans `data/interim/trafic/referentiel_nettoye.parquet` : une ligne par arc gardé, référentiel déjà dédoublonné, avec le tracé (Lambert-93) et les nœuds amont et aval. Pour une carte, il suffit de `trafic.merge(referentiel, on="iu_ac")` : pas de nettoyage du référentiel à refaire.

In [ ]:
# même sortie que `uv run python -m src.clean.trafic`
nettoyage.enregistrer(trafic, arcs, RACINE / CFG["chemins"]["interim"] / "trafic")

## Analyse du dataset

### Satistiques

In [ ]:
# montrer la distribution de débit et présenter moyenne, quartiles, std
display(trafic["q"].describe().to_frame("débit q (véh/h)").T.round(0))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(trafic["q"].dropna(), bins=80, color=BLEU)
ax.set_yscale("log")
ax.axvline(trafic["q"].median(), color=ROUGE, linewidth=1.5, label=f"médiane : {trafic['q'].median():.0f} véh/h")
ax.set_title("La plupart des débits sont sous 2 000 véh/h ; un second groupe vers 5 000 véh/h")
ax.set_xlabel("débit q (véh/h)")
ax.set_ylabel("nombre de lignes (échelle log)")
ax.legend(frameon=False)
plt.show()

In [ ]:
# Distribution de la longueur des axes
longueur = arcs_gardes.length  # en mètres (Lambert-93)
display(longueur.describe().to_frame("longueur (m)").T.round(0))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(longueur, bins=40, color=BLEU)
ax.set_title(f"Un arc gardé mesure {longueur.median():.0f} m en médiane")
ax.set_xlabel("longueur de l'arc (m)")
ax.set_ylabel("nombre d'arcs")
plt.show()

### Tendances

In [ ]:
# visualisation du trafic moyen par heure de la semaine
heure_semaine = trafic["heure"].dt.dayofweek * 24 + trafic["heure"].dt.hour  # 0 = lundi 0 h, 167 = dimanche 23 h
profil = trafic.groupby(heure_semaine)["q"].mean()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(profil.index, profil.values, color=BLEU, linewidth=1.8)
for j in range(1, 7):
    ax.axvline(24 * j, color=GRIS, linewidth=0.8)
ax.set_xticks(range(12, 168, 24), JOURS)
ax.set_xlim(0, 167)
ax.set_title("Le trafic suit le rythme des journées : creux la nuit, plus faible le week-end")
ax.set_xlabel("heure de la semaine (UTC, début de période)")
ax.set_ylabel("débit moyen par arc (véh/h)")
ax.grid(axis="x", visible=False)
plt.show()

In [ ]:
# Visualisation du trafic moyen par axe (figure geopandas)
q_moyen = trafic.groupby("iu_ac")["q"].mean().rename("q_moyen")
carte = arcs_gardes.merge(q_moyen, left_on="iu_ac", right_index=True).sort_values("q_moyen")  # forts débits dessinés en dernier

fig, ax = plt.subplots(figsize=(9, 7))
arcs.plot(ax=ax, color="#e6e5e1", linewidth=0.5)
carte.plot(ax=ax, column="q_moyen", cmap="viridis", norm=LogNorm(), linewidth=1.5, legend=True,
           legend_kwds={"label": "débit moyen (véh/h, échelle log)", "shrink": 0.6})
ax.set_title("Le périphérique et les grands axes concentrent le trafic")
ax.set_axis_off()
plt.show()

In [ ]:
# Même carte avec le taux d'occupation moyen par axe
k_moyen = trafic.groupby("iu_ac")["k"].mean().rename("k_moyen")
carte_k = (arcs_gardes.merge(k_moyen, left_on="iu_ac", right_index=True)
           .sort_values("k_moyen", na_position="first"))  # arcs sans k dessinés en premier
sans_k = carte_k["k_moyen"].isna().sum()

fig, ax = plt.subplots(figsize=(9, 7))
arcs.plot(ax=ax, color="#e6e5e1", linewidth=0.5)
vmax = carte_k["k_moyen"].quantile(0.95)  # quelques arcs très chargés écraseraient l'échelle
carte_k.plot(ax=ax, column="k_moyen", cmap="viridis", vmax=vmax, linewidth=1.5, legend=True,
             legend_kwds={"label": f"taux d'occupation moyen (%, plafonné à {vmax:.0f})", "shrink": 0.6, "extend": "max"},
             missing_kwds={"color": GRIS, "linewidth": 0.8})
ax.legend(handles=[Line2D([0], [0], color=GRIS, lw=2, label=f"arcs gardés sans taux d'occupation ({sans_k})")],
          loc="lower left", frameon=False)
ax.set_title("Le taux d'occupation est le plus fort sur le périphérique et quelques carrefours")
ax.set_axis_off()
plt.show()

### Valeurs manquantes

In [ ]:
# Visualisation des valeurs manquantes du débit par jour de l'année
manquant = trafic["q"].isna().groupby(trafic["heure"].dt.floor("D")).mean() * 100

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(manquant.index, manquant.values, color=BLEU, linewidth=1.5)
ax.axhline(manquant.median(), color=GRIS, linestyle="--", linewidth=1,
           label=f"médiane : {manquant.median():.1f} %".replace(".", ","))
ax.set_ylim(0, 100)
ax.set_xticks(jours[debuts_mois], noms_mois)
ax.set_title("Peu de débits manquent au quotidien, sauf pendant cinq coupures de la collecte")
ax.set_xlabel("jour (UTC)")
ax.set_ylabel("débits manquants (%)")
ax.legend(frameon=False)
plt.show()

print("Jours avec plus de 50 % de débits manquants :", ", ".join(manquant[manquant > 50].index.strftime("%d/%m/%Y")))

# heures incomplètes : à chaque heure, part des arcs gardés dont la valeur manque
par_heure = trafic[["q", "k"]].isna().groupby(trafic["heure"]).mean()
print(f"\nHeures incomplètes, sur {len(par_heure)} heures et {trafic['iu_ac'].nunique()} arcs :")
for colonne, nom in [("q", "débit"), ("k", "taux d'occupation")]:
    print(f"- {nom} : {(par_heure[colonne] > 0).mean():.1%} des heures ont au moins un arc sans valeur, "
          f"{(par_heure[colonne] > 0.1).mean():.1%} en ont plus de 10 % ; "
          f"en médiane, {par_heure[colonne].median():.1%} des arcs n'ont pas de valeur à une heure donnée")

## Bilan du nettoyage

In [ ]:
n_ref = len(ref) - len(arcs)
print(f"Référentiel : {n_ref} versions anciennes retirées sur {len(ref)} lignes ({n_ref / len(ref):.1%})".replace(".", ","))


def en_texte(n):
    """Entier avec espaces comme séparateur de milliers."""
    return f"{n:,}".replace(",", " ")


def variation(apres, avant):
    """Écart entre deux étapes, en nombre et en pourcentage de l'étape précédente."""
    d = apres - avant
    return "0" if d == 0 else f"{d:+,} ({d / avant:+.2%})".replace(",", " ").replace(".", ",")


colonnes = ["lignes", "arcs", "débits q renseignés", "taux k renseignés"]
debut, fin = etapes[0], etapes[-1]
tableau = [{"étape": debut["étape"], **{c: en_texte(debut[c]) for c in colonnes}}]
for avant, apres in zip(etapes, etapes[1:]):
    tableau.append({"étape": apres["étape"], **{c: variation(apres[c], avant[c]) for c in colonnes}})
tableau.append({"étape": "dataset final",
                **{c: f"{en_texte(fin[c])} ({fin[c] / debut[c]:.1%} du brut)".replace(".", ",") for c in colonnes}})
pd.DataFrame(tableau).set_index("étape")

### Ce que fait chaque étape

Dans le tableau, chaque pourcentage est calculé par rapport à l'étape précédente.

1. **Concaténation des 11 fichiers mensuels** : 23 489 735 lignes et 2 980 arcs. Rien n'est retiré.
2. **Préparation** : on ne garde que `iu_ac`, `q` et `k`. `etat_trafic` est déduit de `k`, et `etat_barre = Invalide` correspond toujours à `q` et `k` manquants. `t_1h` (fin de période, UTC) devient `heure` (début de période, UTC), et on ne garde que les heures de la période. Aucune ligne retirée : chaque fichier mensuel contient déjà exactement les heures du mois.
3. **Dédoublonnage du référentiel** : pour chaque arc, on garde la version dont `date_fin`, puis `date_debut`, est la plus récente. 391 lignes du référentiel sont retirées sur 3 739 (10,5 %). Deux arcs (VERG et Souterrain_Etoile) ont deux versions aux mêmes dates, mais ils n'ont aucune donnée de trafic. Aucune ligne de trafic n'est retirée à cette étape.
4. **Arcs sans géométrie** : 31 arcs du trafic n'ont pas de position dans le référentiel, et on ne peut donc pas les rattacher à une station. On retire 31 arcs (1,0 %) et 234 126 lignes (1,00 %), dont 119 217 débits renseignés (0,94 %).
5. **Séries de zéros de plus de 24 h** : un débit nul pendant plus d'une journée est une panne, pas une rue vide. Les valeurs passent à `NaN` mais les lignes restent : 6 049 débits (0,05 %, sur 43 arcs) et 6 799 taux d'occupation (0,05 %, sur 46 arcs).
6. **Seuil de 70 % d'heures avec un débit** : on retire 1 418 arcs (48,1 %) et 11 182 457 lignes (48,08 %). Ces arcs sont surtout des arcs vides : on ne perd que 1 380 502 débits renseignés (11,0 %). On perd en revanche 28,5 % des taux d'occupation, car on écarte des arcs qui mesurent `k` sans `q`.
7. **Grille complète** : on ajoute 199 344 lignes (+1,7 %) pour les heures absentes du fichier (coupures de la collecte), avec `q` et `k` à `NaN`. Aucune valeur n'est ajoutée.
8. **Informations des arcs** : chaque ligne reçoit le libellé, la longueur (`longueur_m`) et le centroïde (`x_centroide`, `y_centroide`, en Lambert-93) de son arc. Aucune ligne n'est ajoutée ni retirée.

Au final, le dataset compte 12 272 496 lignes (52,2 % du brut) et 1 531 arcs (51,4 %). Il conserve 88,2 % des débits renseignés du brut. Il manque 8,6 % des débits et 23,2 % des taux d'occupation.